# Jane Street — 500-date training on Colab GPU

The Mac CPU capped us at 280 training dates (16 GB RAM + OMP scaling). This notebook trains `lstm_modelr`, `gru_modelr`, and (optionally) `mamba` on **500 training dates** using a Colab T4/A100 GPU.

## Setup

1. Runtime → Change runtime type → **GPU** (T4 is fine; A100 is faster).
2. Upload the Jane Street parquet data (see the *Data upload* section below).
3. Run each cell in order.

Everything below expects Python ≥ 3.11 (Colab default in 2026).


## 1. Data upload

The Jane Street data lives on Kaggle. Fastest options (pick one):

**Option A — Kaggle API (recommended, ~2 min).** Upload your `kaggle.json` credentials, then:
```
!pip install -q kaggle
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle competitions download -c jane-street-real-time-market-data-forecasting -p /content/data
!cd /content/data && unzip -q '*.zip'
```

**Option B — Google Drive.** Upload the `data/` folder to your Drive once, then mount:
```
from google.colab import drive
drive.mount('/content/drive')
!ln -sf /content/drive/MyDrive/JaneStreetData /content/data
```

The training script expects the same layout as our local `data/`: `data/train.parquet/`, `data/features.csv`, `data/responders.csv`, etc.


In [ ]:
# Verify GPU + data
!nvidia-smi
!ls -la /content/data/ | head -10

## 2. Clone the project + install deps

Two options: (a) push the project to a git repo you own and clone it here, or (b) upload it as a zip and unzip.

For option (a):
```
!git clone https://github.com/YOUR_USERNAME/JaneStreetChallenge.git /content/project
%cd /content/project
```

For option (b) — after uploading `project.zip` via the Colab file browser:
```
!unzip -q project.zip -d /content/project
%cd /content/project
```


In [ ]:
# Install project deps into the Colab environment.
# uv works on Colab but pip is simpler for a one-shot.
%pip install --quiet \
  'einops>=0.8' 'numpy>=2.0' 'polars>=1.20' 'pyarrow>=15' \
  'scikit-learn>=1.5' 'torch>=2.4' 'tqdm>=4.66' 'xgboost>=2.1' \
  'iisignature>=0.24'

# Install the project itself in editable mode.
%pip install --quiet -e /content/project

In [ ]:
# Verify torch sees the GPU
import torch
print(f'torch    : {torch.__version__}')
print(f'cuda ok  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'device   : {torch.cuda.get_device_name(0)}')
    print(f'mem (GB) : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}')

## 3. Point the config at the Colab data path

The default `Cfg.data_root` is our local Mac path. Override it inline.

In [ ]:
import os, warnings
warnings.filterwarnings('ignore')

os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'    # same libomp guard we use locally

from pathlib import Path
from janestreet.config import Cfg

cfg = Cfg()
cfg.data_root = Path('/content/data')          # <- Colab path
cfg.artifacts_root = Path('/content/artifacts')
cfg.artifacts_root.mkdir(exist_ok=True, parents=True)

## 4. Load 500 train dates + 100 valid tail

We pin the same validation window (dates 1599–1698) so results are directly comparable to the local run.

In [ ]:
import time, gc
import numpy as np
import polars as pl

from janestreet.config import COL_DATE
from janestreet.pipeline import prepare_dataset, make_pipeline

# 600 total → 500 train (1099..1598) + 100 valid (1599..1698)
cfg.min_date_id = 1099
cfg.max_date_id = 1698
N_VALID = 100

t0 = time.time()
df = prepare_dataset(cfg)
print(f'loaded {df.height:,} rows × {df.width} cols in {time.time()-t0:.1f}s')

dates = np.sort(df.select(pl.col(COL_DATE).unique()).to_series().to_numpy())
valid_dates = dates[-N_VALID:]
train_dates = dates[:-N_VALID]
print(f'train: {train_dates[0]}..{train_dates[-1]} ({len(train_dates)} dates)')
print(f'valid: {valid_dates[0]}..{valid_dates[-1]} ({len(valid_dates)} dates)')

# Split up-front & free the full frame (same memory hygiene as bench.py).
df_train = df.filter(pl.col(COL_DATE).is_in(train_dates))
online_dates = np.concatenate([train_dates[-1:], valid_dates])
df_valid_plus = df.filter(pl.col(COL_DATE).is_in(online_dates))
df_va = df_valid_plus.filter(pl.col(COL_DATE).is_in(valid_dates))
del df; gc.collect()
print(f'rows retained: train={df_train.height:,}  valid+1={df_valid_plus.height:,}')

## 5. Train lstm_modelr on 500 dates (GPU)

This is the run that matters. On a T4 it should take ~1.5 h; on an A100 ~30 min. Compare to our Mac CPU where lstm_modelr on 280 dates took 4 h.

In [ ]:
import time
# ── Cold-start training with per-epoch Drive checkpointing ─────────────
# If you use section 5b (warm-start) instead, skip this cell.
#
# We save a full training checkpoint to Google Drive after every epoch so
# a Colab disconnect doesn't cost you the run. Each fit_state_*.pt is
# ~15 MB and holds model + optimizer + best-so-far — everything needed
# to resume mid-fit.

# Mount Drive so checkpoints persist across runtime restarts.
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    CKPT_DIR = Path('/content/drive/MyDrive/JS_ckpts/lstm_500d')
except Exception as e:
    print(f'Drive mount failed ({e!r}) — falling back to ephemeral /content/. '
          f'You WILL lose progress on disconnect.')
    CKPT_DIR = Path('/content/ckpts/lstm_500d')
CKPT_DIR.mkdir(parents=True, exist_ok=True)
print(f'checkpoints will land at: {CKPT_DIR}')

cfg.model_name = 'lstm_modelr'
cfg.model_kwargs = dict(
    hidden_sizes=[96, 96, 96], dropout_rates=[0.1, 0.1, 0.1],
    lr=1e-3, weight_decay=1e-2,
    epochs=15, batch_size=1, grad_clip=1.0,
    early_stopping_patience=4, lr_refit=1e-3,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    num_aux=4,
)

pipe = make_pipeline(cfg)
# Snapshot the full pipeline once at the start so we have preprocessor +
# feature schema on disk. The per-epoch .pt files below only carry the
# training state; you need both to fully resume.
pipe.save(CKPT_DIR / 'pipeline.pkl')

t0 = time.time()
pipe.fit(df_train, df_va, verbose=True, epoch_save_dir=CKPT_DIR)
print(f'\nfit time: {(time.time()-t0)/60:.1f} min')
print(f'\ncheckpoints in {CKPT_DIR}:')
!ls -la {CKPT_DIR}


## 5b. (Alternative) Warm-start from your local 280-date checkpoint

Skip section 5's cold-start and use this instead if you'd like to reuse the local `lstm_modelr` weights already trained on 280 dates. You'll fine-tune with a smaller LR for a few epochs — typically saves ~60% of the fit time vs. cold-start on the same data.

**Files to upload to Colab first** (via the left sidebar file browser, into `/content/`):

* `artifacts/bench/run2_280d/checkpoints/lstm_modelr.pkl`   (11 MB — the model)
* `artifacts/bench/run2_280d/checkpoints/lstm_modelr.pkl.meta` (feature schema + preprocessor)

Both files must be uploaded together; `FullPipeline.load` reads them as a pair.

Semantics:
* **Preprocessor is preserved** (running-mean/std of features fit on the 280-date train slice). Slight staleness vs. re-fitting on 500 dates is fine — the standardization coefficients don't shift meaningfully with more data.
* **Adam's running moments are lost** through the pickle. Start with a smaller LR to avoid the first few steps blowing up before Adam re-estimates its second moments — 2e-4 is a safe default (Phase D showed the LR plateau is 3e-4–1e-3; we go to the low end after warm-start).
* Architecture kwargs (`hidden_sizes`, `signature_channels`, …) are baked into the loaded weights — do not modify them at warm-start time.

If you use this cell, **skip cell 11** (the cold-start).


In [ ]:
from janestreet.pipeline import FullPipeline

ckpt = Path('/content/lstm_modelr.pkl')
assert ckpt.exists(), 'upload lstm_modelr.pkl AND lstm_modelr.pkl.meta into /content/ first'
assert ckpt.with_suffix(ckpt.suffix + '.meta').exists(), 'meta sidecar missing'

pipe = FullPipeline.load(ckpt)
# Move the model to GPU if available (the checkpoint was CPU-trained).
device = 'cuda' if torch.cuda.is_available() else 'cpu'
pipe.model.device = torch.device(device)
if pipe.model.model is not None:
    pipe.model.model = pipe.model.model.to(device)

print(f'loaded  model    = {type(pipe.model).__name__}')
print(f'        device   = {pipe.model.device}')
print(f'        n_params = {sum(p.numel() for p in pipe.model.model.parameters()):,}')
print(f'        features = {len(pipe.feature_cols)}')

# Warm-start fine-tune: 5 more epochs on the 500-date frame at a smaller LR.
pipe.model.lr = 2e-4                # was 1e-3 for the 280-date cold-start
pipe.model.epochs = 5
pipe.model.early_stopping_patience = 3

t0 = time.time()
pipe.fit(df_train, df_va, verbose=True, warm_start=True)
print(f'\nwarm-start fine-tune time: {(time.time()-t0)/60:.1f} min')


## 5c. (Recovery) Resume from a mid-training Drive checkpoint

Use this if Colab disconnected during cell 11's fit, and you want to pick up where you left off. Requires cell 11 to have written at least one `epoch_XXX.pt` to Drive.

Semantics differ from warm-start (5b) in the important way: **AdamW state, epoch counter, and best-so-far metric are ALL restored** — you lose zero training progress. This is exactly what warm-start couldn't do.


In [ ]:
from janestreet.pipeline import FullPipeline

# 1. Re-mount Drive (safe to re-run — no-op if already mounted).
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
CKPT_DIR = Path('/content/drive/MyDrive/JS_ckpts/lstm_500d')
assert (CKPT_DIR / 'pipeline.pkl').exists(), (
    f'no pipeline.pkl in {CKPT_DIR} — cell 11 never ran, nothing to resume from'
)
assert (CKPT_DIR / 'latest.pt').exists(), (
    f'no latest.pt in {CKPT_DIR} — cell 11 must have crashed before finishing epoch 1'
)

# 2. Rebuild the pipeline (preprocessor + feature schema + model architecture).
pipe = FullPipeline.load(CKPT_DIR / 'pipeline.pkl')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
pipe.model.device = torch.device(device)

# 3. Resume: reads latest.pt to restore model + AdamW + epoch counter + best-so-far.
#    Keeps writing new checkpoints to the same dir so a further disconnect is also safe.
t0 = time.time()
pipe.fit(
    df_train, df_va, verbose=True,
    epoch_save_dir=CKPT_DIR,
    resume_from=CKPT_DIR / 'latest.pt',
)
print(f'\nresume time: {(time.time()-t0)/60:.1f} min')


## 6. Walk-forward evaluation (static + online refit)

Same protocol as `scripts/bench.py`. Emits per-day predictions we save as `.npz` for the local ensemble.

In [ ]:
import copy
from janestreet.config import COL_WEIGHT
from janestreet.training.metrics import r2_weighted

target_col = pipe.target_col

def walk_forward(pipe, df_all, valid_dates):
    pipe = copy.deepcopy(pipe)
    preds, ys, ws = [], [], []
    for i, dt in enumerate(valid_dates):
        day = df_all.filter(pl.col(COL_DATE) == dt)
        if i > 0:
            prev = df_all.filter(pl.col(COL_DATE) == int(dt) - 1)
            if prev.height > 0:
                pipe.update(prev)
        preds.append(pipe.predict(day))
        ys.append(day.select(target_col).to_series().to_numpy())
        ws.append(day.select(COL_WEIGHT).to_series().to_numpy())
    return np.concatenate(preds), np.concatenate(ys), np.concatenate(ws)

# Static: predict each valid day, no refit.
static = copy.deepcopy(pipe)
if hasattr(static.model, 'lr_refit'):
    static.model.lr_refit = 0.0
p_s, y_s, w_s = walk_forward(static, df_valid_plus, valid_dates)
print(f'static R² = {r2_weighted(y_s, p_s, w_s):+.5f}')

# Online refit at lr=1e-3 (Phase D winner for lstm_modelr).
pipe.model.lr_refit = 1e-3
p_o, y_o, w_o = walk_forward(pipe, df_valid_plus, valid_dates)
print(f'online R² @ lr_refit=1e-3 = {r2_weighted(y_o, p_o, w_o):+.5f}')

In [ ]:
# Dump .npz files so you can pull them back to your Mac and blend them into the
# existing ensemble (same shape/format as the bench's per-model preds).
out_dir = cfg.artifacts_root / 'colab_500d' / 'preds'
out_dir.mkdir(parents=True, exist_ok=True)

np.savez_compressed(out_dir / 'lstm_modelr_500_static.npz',
    preds=p_s.astype(np.float32), y=y_s.astype(np.float32), w=w_s.astype(np.float32),
    valid_dates=np.asarray(valid_dates, dtype=np.int32))
np.savez_compressed(out_dir / 'lstm_modelr_500_online.npz',
    preds=p_o.astype(np.float32), y=y_o.astype(np.float32), w=w_o.astype(np.float32),
    valid_dates=np.asarray(valid_dates, dtype=np.int32))

print('written:')
!ls -la {out_dir}

## 7. Download the preds back to your Mac

Colab's file browser (left sidebar) → navigate to `/content/artifacts/colab_500d/preds/` → right-click each `.npz` → **Download**. Then drop them into `artifacts/bench/run3_colab500/preds/` on your Mac and blend with the existing streams:

```
uv run python scripts/ensemble_blend.py \
    --pred artifacts/bench/run2_280d/preds/xgb_static.npz \
    --pred artifacts/bench/run2_280d/preds/gru_modelr_online.npz \
    --pred artifacts/bench/run3_colab500/preds/lstm_modelr_500_online.npz \
    --out artifacts/bench/run3_colab500/ensemble_3way_lstm500.json
```


## 8. Optional: also train gru_modelr and mamba on 500 dates

Adds ~1.5-3 h more; both dump their own .npz. Uncomment to run.

In [ ]:
# for name, kw in [
#     ('gru_modelr', dict(
#         hidden_sizes=[96, 96, 96], dropout_rates=[0.1, 0.1, 0.1],
#         lr=1e-3, weight_decay=1e-2,
#         epochs=15, batch_size=1, grad_clip=1.0,
#         early_stopping_patience=4, lr_refit=1e-3,
#         device='cuda' if torch.cuda.is_available() else 'cpu', num_aux=4,
#     )),
#     ('mamba', dict(
#         d_model=96, n_layers=3, d_state=16, d_conv=4, expand=2, dropout=0.1,
#         lr=5e-4, weight_decay=1e-2,
#         epochs=12, batch_size=1, grad_clip=1.0,
#         early_stopping_patience=4, lr_refit=1e-3,
#         device='cuda' if torch.cuda.is_available() else 'cpu',
#     )),
# ]:
#     print(f'\n=== {name} ===')
#     cfg.model_name = name; cfg.model_kwargs = kw
#     pipe = make_pipeline(cfg)
#     t0 = time.time(); pipe.fit(df_train, df_va, verbose=True)
#     print(f'fit: {(time.time()-t0)/60:.1f} min')
#     p_o, y_o, w_o = walk_forward(pipe, df_valid_plus, valid_dates)
#     print(f'{name} online R² = {r2_weighted(y_o, p_o, w_o):+.5f}')
#     np.savez_compressed(out_dir / f'{name}_500_online.npz',
#         preds=p_o.astype(np.float32), y=y_o.astype(np.float32), w=w_o.astype(np.float32),
#         valid_dates=np.asarray(valid_dates, dtype=np.int32))